In [7]:
#setup
%load_ext autoreload
%autoreload 2

from datetime import date
from datetime import datetime
import netCDF4 as nc
import os
import pandas as pd
import sys
import time
import fiona
import xarray as xr

sys.path.append('D:\\repos\\E-OBS-SWB2\\Python')

from EOBSobject import EOBSobject
from RechargeCalc import RechargeCalc

In [4]:
cwd = 'D:/Dati pesanti/SWB2_MAURICE'
cwd

'D:/Dati pesanti/SWB2_MAURICE'

## Use EOBSobject

In [10]:
outpath = os.path.join(cwd, 'climate_ncfile')
inpath = os.path.join(cwd, 'data_original', 'E-OBS')

In [ ]:
vars = ['rr', 'tn', 'tx']
outnames = ['prcp', 'tmax', 'tmin']

coord = {'lon': [8.691, 8.929, 9.524, 9.537],
            'lat': [45.611, 45.308, 45.610, 45.306]}
coord = pd.DataFrame(coord)
# need one file per year
start = 2019
end = 2024

In [ ]:
for i, var in enumerate(vars):
    f = EOBSobject(var, inpath, outpath, folder = True, swb2 = True)
    f.load()
    # cut in space and time
    f.set_outname(outnames[i])
    f.cut_spacetime(coord, start, end, option = 'singleyear', contourcell=2, autosave = True)
    f.close_netcdf()

### Cut the result in time

In [11]:
# load swb2 output and cut it
# from jan 2019 to dec 2019
# and in the MAURICE area
swb2path = os.path.join(cwd, 'output', 'ModelMI_net_infiltration__2019-01-01_2024-12-31__338_by_660.nc')
outswb2 = xr.open_dataset(swb2path, engine='netcdf4')

In [12]:
outswb2

<xarray.Dataset> Size: 2GB
Dimensions:           (time: 2192, y: 338, x: 660)
Coordinates:
  * time              (time) datetime64[ns] 18kB 2019-01-01 ... 2024-12-31
  * y                 (y) float64 3kB 5.051e+06 5.051e+06 ... 5.017e+06
  * x                 (x) float64 5kB 1.476e+06 1.476e+06 ... 1.542e+06
    lat               (y, x) float64 2MB ...
    lon               (y, x) float64 2MB ...
Data variables:
    net_infiltration  (time, y, x) float32 2GB ...
    crs               int32 4B ...
Attributes:
    source:              net_infiltration output from SWB run started on May ...
    executable_version:  version 2.0 Beta, Git branch:  master, Git commit ha...
    conventions:         CF-1.6
    history:             May 14 2025 16:35:16: Soil-Water-Balance run started.

In [14]:
datetime(2019,1,1) - datetime(2019, 12, 31)

datetime.timedelta(days=-364)

In [19]:
cut = outswb2.isel(time=slice(0, 364+1)).copy()

In [20]:
cut

<xarray.Dataset> Size: 329MB
Dimensions:           (time: 365, y: 338, x: 660)
Coordinates:
  * time              (time) datetime64[ns] 3kB 2019-01-01 ... 2019-12-31
  * y                 (y) float64 3kB 5.051e+06 5.051e+06 ... 5.017e+06
  * x                 (x) float64 5kB 1.476e+06 1.476e+06 ... 1.542e+06
    lat               (y, x) float64 2MB ...
    lon               (y, x) float64 2MB ...
Data variables:
    net_infiltration  (time, y, x) float32 326MB ...
    crs               int32 4B ...
Attributes:
    source:              net_infiltration output from SWB run started on May ...
    executable_version:  version 2.0 Beta, Git branch:  master, Git commit ha...
    conventions:         CF-1.6
    history:             May 14 2025 16:35:16: Soil-Water-Balance run started.

In [21]:
cut.to_netcdf(os.path.join(cwd, 'output', 'ModelMI_net_infiltration__2019-01-01_2019-12-31__338_by_660.nc'))

In [22]:
check = xr.load_dataset(os.path.join(cwd, 'output', 'ModelMI_net_infiltration__2019-01-01_2019-12-31__338_by_660.nc'), format = 'netcdf4')

## Use RechargeCalc

In [5]:
start = time.time()

cell_area = 100*100 #m2
#Path to the SWB2 output
swb2path = os.path.join(os.path.join(cwd, 'output', 'ModelMI_net_infiltration__2019-01-01_2019-12-31__338_by_660.nc'))
#Path to the input .csv files folder
inputpath = os.path.join(cwd,'data_original', 'file_input_rechargecalc', 'swb_MODELMI19')
sppath = os.path.join(inputpath, 'rirrigua_speciale_swb_MODELMI19.csv')

r = RechargeCalc(cell_area, uniqueid = 'indicatore', nSP = 4)
r.load_inputfiles(swb2path, inputpath)

SP1 = 90   #days, 01/01 - 30/03
SP2 = 76   #days, 01/04 - 12/06
SP3 = 92   #days, 13/06 - 15/09
SP4 = 107  #days, 16/09 - 31/12
SPs = [SP1, SP2, SP3, SP4]

r.meteoricR(SPs, units = 'ms', fixrow=1, fixcol=4)

coeffs = {
    'E': 0.3,  #Irrigation technique efficiency
    'R': 0.05, #Residual runoff
    'RISP': 1, #1 - fraction of water saved by a change of irrigation technique
    'P': 1     #Percentage of the cell covered by the irrigation
    }

col = ['land_cover', 'land_cover', 'zona_urbana']
valcol = [123, 124, 1]
option = [0, 1] #0: OR, 1: AND

r.urbanR(coeff=0.125, col=col, valcol=valcol, option=option)

r.irrigationR(coeffs, specialpath=sppath)

r.totalR(fillna=True)

# r.export('recharge','rtot',
#              outpath = os.path.join(cwd, 'rtot'),
#              outname = 'rtot_swb_MODELMI19',
#              withcoord=True,
#              coordpath = os.path.join(inputpath, 'coord.csv'))
# r.georef('recharge','rtot',
#             outpath = os.path.join(cwd, 'rtot'),
#             fname = 'rtot_swb_MODELMI19.shp', 
#             coordpath = os.path.join(inputpath, 'coord.csv'),
#             crs = 'epsg:3003', dropcoord=False, driver = 'ESRI Shapefile')
end = time.time()
print((end - start)/60, 'min')

Loading the input files
-----------------------
indicatori file found
ricarica_irrigua file found
extractions file found
Meteoric recharge dataframe creation
------------------------------------
Performing the sum of net_infiltration over the stress periods provided
Output unit measure: ms
End of the procedure
Elapsed time: 2.5 s
Urban recharge dataframe creation
---------------------------------
Elapsed time: 5.43 s
Irrigation recharge dataframe creation
--------------------------------------
Elapsed time: 0.66 s
Total recharge dataframe creation
---------------------------------
rmeteo :  65.35254992974421
rirr :  30.417757932403887
rurb :  4.2296921378518775
Elapsed time: 0.59 s
0.17158968051274617 min


In [6]:
r.get_df('recharge', 'rmeteo').iloc[:,1:21].sum()

nrow    3.758898e+07
ncol    7.350486e+07
SP1     3.499805e-03
SP2     5.279571e-03
SP3     3.456238e-03
SP4     8.211318e-03
dtype: float64

In [ ]:
r.get_df('recharge', 'rmeteo').iloc[:, 3:].sum()


SP1    0.003500
SP2    0.005280
SP3    0.003456
SP4    0.008211
dtype: float64

In [52]:
r.get_df('recharge', 'rirr').iloc[:, 3:].sum()

SP1    0.000000
SP2    0.003290
SP3    0.004018
SP4    0.000000
dtype: float64

In [53]:
r.get_df('recharge', 'rurb').iloc[:, 4:].sum()

SP1    0.000252
SP2    0.000254
SP3    0.000257
SP4    0.000253
dtype: float64

In [55]:
r.get_df('recharge', 'rtot').iloc[:, 3:].sum()

SP1    0.002947
SP2    0.007624
SP3    0.006932
SP4    0.006524
dtype: float64

In [56]:
rtotcheck = pd.read_csv(os.path.join(cwd, 'rtot', 'rtot_IRR0_2050.csv'))

In [59]:
rtotcheck.iloc[:, 5:25].sum()

SP1     0.003936
SP2     0.004463
SP3     0.004444
SP4     0.003537
SP5     0.001661
SP6     0.004203
SP7     0.004284
SP8     0.000659
SP9     0.001948
SP10    0.004183
SP11    0.004354
SP12    0.001017
SP13    0.000746
SP14    0.004135
SP15    0.004414
SP16    0.001215
SP17    0.001624
SP18    0.004609
SP19    0.004292
SP20    0.002076
dtype: float64